# Jyotish assistant: live demo on a T4

Runs the GGUF with llama.cpp and the full pipeline behind a Gradio page with a temporary public link and a password. The session ends after `DEMO_HOURS`.

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, tarfile, time, urllib.request, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # one GPU: llama.cpp would otherwise split the model across both T4s

# ---------- settings ----------
RELEASE = "b11513"
MODEL_FILE = "astro-v2-8b.Q4_K_M.gguf"
PARALLEL = 4                  # simultaneous requests the server handles
CTX_PER_SLOT = 5120           # a chart prompt is about 2,800 tokens and an answer up to 1,000
PORT = 8080
WORK = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
# prebuilt Linux CUDA 12.8 build of llama.cpp and its CUDA runtime, with the sha256 GitHub publishes for release b11513
PREBUILT = {
    "llama-b11513-bin-ubuntu-cuda-12.8-x64.tar.gz": "af399b958079ec0a0a002faa5b2432f63269c0bf862e9f4f8add67a38a17aa33",
    "cudart-llama-b11513-bin-ubuntu-cuda-12.8-x64.tar.gz": "bb8ac07d9ae02a9d834d03cd5b2ed7c3b9720f46aea02414d0c304d066fec13c",
}

DEMO_HOURS = 1.5              # the page stays up this long, then the session ends and stops using GPU quota

In [ ]:
!pip install -q gradio pyswisseph scikit-learn==1.9.1 scipy==1.17.1 joblib httpx

In [ ]:
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr)
    print(out[-2500:], flush=True)
    if check:
        assert r.returncode == 0, "command failed: " + cmd
    return r.returncode, out

print(sh("nvidia-smi")[1][:900])
MODEL = glob.glob("/kaggle/input/**/" + MODEL_FILE, recursive=True)
assert MODEL, MODEL_FILE + " not found: attach the output of the astro-export kernel"
MODEL = MODEL[0]
print("model:", MODEL, os.path.getsize(MODEL), "bytes")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def install_prebuilt():
    d = WORK + "/llama_prebuilt"
    os.makedirs(d, exist_ok=True)
    for name, digest in PREBUILT.items():
        path = d + "/" + name
        urllib.request.urlretrieve("https://github.com/ggml-org/llama.cpp/releases/download/%s/%s" % (RELEASE, name), path)
        got = sha256(path)
        assert got == digest, "sha256 mismatch for " + name + ": " + got
        with tarfile.open(path) as t:
            t.extractall(d)
        os.remove(path)
    server = glob.glob(d + "/**/llama-server", recursive=True)[0]
    bench = glob.glob(d + "/**/llama-bench", recursive=True)[0]
    libs = sorted({os.path.dirname(p) for p in glob.glob(d + "/**/*.so*", recursive=True)})
    return server, bench, ":".join(libs)

def build_from_source():
    src = WORK + "/llama.cpp"
    sh("git clone --depth 1 --branch %s https://github.com/ggml-org/llama.cpp %s" % (RELEASE, src))
    sh("cd %s && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF > /dev/null "
       "&& cmake --build build --target llama-server llama-bench -j4" % src)
    return src + "/build/bin/llama-server", src + "/build/bin/llama-bench", src + "/build/bin"

def works(bench, libs):
    env = "LD_LIBRARY_PATH=%s:$LD_LIBRARY_PATH " % libs
    rc, out = sh(env + "%s -m %s -ngl 99 -p 64 -n 8 -r 1" % (bench, MODEL), check=False)
    return rc == 0 and "CUDA" in out and "tg8" in out

try:
    SERVER, BENCH, LIBS = install_prebuilt()
    ok = works(BENCH, LIBS)
except Exception as e:
    print("prebuilt build failed:", repr(e)[:300])
    ok = False
if not ok:
    print("prebuilt build did not run on this GPU/driver: building llama.cpp from source (about 10 minutes)")
    SERVER, BENCH, LIBS = build_from_source()
    assert works(BENCH, LIBS), "llama.cpp does not run on this machine"
print("llama.cpp ready:", SERVER)

In [ ]:
import requests
os.environ["LD_LIBRARY_PATH"] = LIBS + ":" + os.environ.get("LD_LIBRARY_PATH", "")
server_log = open("/kaggle/working/llama_server.log", "w")
server = subprocess.Popen([SERVER, "-m", MODEL, "-ngl", "99", "-c", str(PARALLEL * CTX_PER_SLOT), "--parallel", str(PARALLEL),
                           "--host", "127.0.0.1", "--port", str(PORT), "-t", "4"], stdout=server_log, stderr=server_log)
BASE = "http://127.0.0.1:%d" % PORT
t0 = time.time()
while True:
    try:
        if requests.get(BASE + "/health", timeout=2).json().get("status") == "ok":
            break
    except Exception:
        pass
    assert server.poll() is None, "llama-server exited: see /kaggle/working/llama_server.log"
    assert time.time() - t0 < 600, "llama-server did not become ready"
    time.sleep(3)
print("server ready after %.0f s" % (time.time() - t0))

def render_qwen(messages, thinking=False):
    # Qwen3 chat format: same text as tokenizer.apply_chat_template(..., enable_thinking=thinking). With thinking off the
    # empty think block is pre-filled; with it on the assistant turn is left open and the model writes <think>...</think> itself.
    text = "".join("<|im_start|>%s\n%s<|im_end|>\n" % (m["role"], m["content"]) for m in messages)
    return text + ("<|im_start|>assistant\n" if thinking else "<|im_start|>assistant\n<think>\n\n</think>\n\n")

In [ ]:
# the pipeline code (chart engine, safety layer, prompts) comes from the astro-app-code dataset
hits = glob.glob("/kaggle/input/**/src/app/pipeline.py", recursive=True)
if not hits:                                   # the dataset was uploaded as a zip: unpack it
    import zipfile
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall(WORK + "/app_code")
    hits = glob.glob(WORK + "/app_code/**/src/app/pipeline.py", recursive=True)
assert hits, "astro-app-code dataset not found or empty"
root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))      # .../src/app/pipeline.py -> the folder holding src/ and prompts/
assert os.path.exists(root + "/prompts/system_v1.txt"), "prompts/system_v1.txt missing from the code bundle"
sys.path.insert(0, root)
os.chdir("/kaggle/working")        # writable; the code folder under /kaggle/input is read-only and Gradio writes .gradio into the cwd
from src.app.pipeline import Pipeline, Birth, DISCLAIMER, llamacpp_llm, find_city
from src.app import safety
v = safety.check_input("I want to end my life")
print("safety layer test:", v.action, v.category, "|", v.reason)
safety._classify("what is the exaltation sign of venus")      # forces the classifier model to load now
print("classifier loaded:", bool(safety._CLF))
import csv
CITIES = sorted(r["city"] for r in csv.DictReader(open(root + "/src/chart/cities.csv", encoding="utf-8")))
pipe = Pipeline(llamacpp_llm(BASE, temperature=0.0, max_tokens=1500))

In [ ]:
import secrets
from datetime import datetime
import gradio as gr

USER, PASSWORD = "astro", secrets.token_urlsafe(9)

def respond(question, use_chart, date_str, time_str, city):
    t0 = time.time()
    try:
        birth = Birth(when=datetime.fromisoformat("%sT%s" % (date_str.strip(), time_str.strip())), city=city) if use_chart else None
        res = pipe.ask(question, birth)
    except ValueError as e:
        return "Please check the birth details: %s" % e, "input error", "", DISCLAIMER
    return res.reply, res.route, "%.1f s" % (time.time() - t0), DISCLAIMER

with gr.Blocks(title="Jyotish assistant") as demo:
    gr.Markdown("## Jyotish assistant\nAsk a question. To use a birth chart, tick the box and fill in the birth details (local time).")
    q = gr.Textbox(label="Question", lines=3)
    with gr.Row():
        use = gr.Checkbox(label="Use a birth chart", value=False)
        d = gr.Textbox(label="Birth date (YYYY-MM-DD)", value="1990-05-17")
        t = gr.Textbox(label="Birth time (HH:MM, local)", value="06:30")
        c = gr.Dropdown(CITIES, label="Birth city", value="Delhi")
    go = gr.Button("Ask")
    reply = gr.Textbox(label="Reply", lines=14)
    with gr.Row():
        route = gr.Textbox(label="Safety route")
        took = gr.Textbox(label="Time")
    note = gr.Textbox(label="Note", lines=3)
    go.click(respond, [q, use, d, t, c], [reply, route, took, note])

res = demo.launch(share=True, auth=(USER, PASSWORD), prevent_thread_lock=True, quiet=True)
share_url = getattr(demo, "share_url", None) or (res[2] if isinstance(res, (tuple, list)) and len(res) > 2 else None)
if not share_url:
    print("WARNING: no public link was created (Gradio could not open a share tunnel). The page only works inside this notebook.")
print("=" * 70)
print("PUBLIC LINK :", share_url)
print("USER        :", USER)
print("PASSWORD    :", PASSWORD)
print("Ends in %.1f hours. Anyone with the link AND the password can use it." % DEMO_HOURS)
print("=" * 70, flush=True)
end = time.time() + DEMO_HOURS * 3600
while time.time() < end:
    time.sleep(30)
print("demo time is up")
server.terminate()